# 02 - Transformasi dan Validasi PySpark
Gunakan JDK 17 pada lokal. Di Colab, jalankan sel instalasi bila PySpark belum tersedia.

In [ ]:
# Khusus Colab bila diperlukan:
# !pip -q install pyspark==3.5.5

In [ ]:
from pathlib import Path
from pyspark.sql import SparkSession, functions as F, types as T

spark = (SparkSession.builder.appName('BigDataUMKMAnalytics').master('local[*]')
         .config('spark.sql.shuffle.partitions', '8').getOrCreate())
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
CSV_PATH = ROOT / 'data' / 'synthetic_umkm_data.csv'
if not CSV_PATH.exists(): CSV_PATH = Path('/content/synthetic_umkm_data.csv')
print('Spark:', spark.version, '| CSV:', CSV_PATH)

In [ ]:
# Schema sumber eksplisit
schema = T.StructType([
    T.StructField('ID', T.LongType(), False),
    T.StructField('Monthly_Revenue', T.LongType(), True),
    T.StructField('Net_Profit_Margin (%)', T.DoubleType(), True),
    T.StructField('Burn_Rate_Ratio', T.DoubleType(), True),
    T.StructField('Transaction_Count', T.IntegerType(), True),
    T.StructField('Avg_Historical_Rating', T.DoubleType(), True),
    T.StructField('Review_Text', T.StringType(), True),
    T.StructField('Review_Volatility', T.DoubleType(), True),
    T.StructField('Business_Tenure_Months', T.IntegerType(), True),
    T.StructField('Repeat_Order_Rate (%)', T.DoubleType(), True),
    T.StructField('Digital_Adoption_Score', T.DoubleType(), True),
    T.StructField('Peak_Hour_Latency', T.StringType(), True),
    T.StructField('Location_Competitiveness', T.IntegerType(), True),
    T.StructField('Sentiment_Score', T.DoubleType(), True),
    T.StructField('Class', T.StringType(), True),
])
raw = spark.read.option('header', True).schema(schema).csv(str(CSV_PATH))
clean_names = ['umkm_id','monthly_revenue','net_profit_margin_pct','burn_rate_ratio','transaction_count','avg_historical_rating','review_text','review_volatility','business_tenure_months','repeat_order_rate_pct','digital_adoption_score','peak_hour_latency','location_competitiveness','sentiment_score','business_class']
umkm = raw.toDF(*clean_names)
print('Rows:', umkm.count()); umkm.printSchema(); umkm.show(3, truncate=False)

In [ ]:
# Transformasi bisnis
transformed = (umkm
    .withColumn('estimated_monthly_profit', F.round(F.col('monthly_revenue') * F.col('net_profit_margin_pct') / 100, 2))
    .withColumn('estimated_monthly_cost', F.round(F.col('monthly_revenue') - F.col('estimated_monthly_profit'), 2))
    .withColumn('revenue_per_transaction', F.round(F.col('monthly_revenue') / F.col('transaction_count'), 2))
    .withColumn('tenure_years', F.round(F.col('business_tenure_months') / 12, 2))
    .withColumn('tenure_band', F.when(F.col('business_tenure_months') < 24, '<2 tahun').when(F.col('business_tenure_months') < 60, '2-5 tahun').when(F.col('business_tenure_months') < 120, '5-10 tahun').otherwise('>=10 tahun'))
    .withColumn('digital_adoption_band', F.when(F.col('digital_adoption_score') < 3, 'Rendah').when(F.col('digital_adoption_score') < 6, 'Menengah').when(F.col('digital_adoption_score') < 8, 'Tinggi').otherwise('Lanjutan'))
    .withColumn('sentiment_band', F.when(F.col('sentiment_score') < 0, 'Negatif').when(F.col('sentiment_score') > 0, 'Positif').otherwise('Netral'))
    .withColumn('operational_risk', F.when((F.col('net_profit_margin_pct') < 0) & (F.col('burn_rate_ratio') > 1), 'High').when((F.col('net_profit_margin_pct') < 0) | (F.col('burn_rate_ratio') > 1), 'Medium').otherwise('Low')))
transformed.select('umkm_id','estimated_monthly_profit','tenure_band','digital_adoption_band','operational_risk').show(5)

In [ ]:
# Quality gates PySpark
row_count = transformed.count()
duplicate_ids = transformed.groupBy('umkm_id').count().filter(F.col('count') > 1).count()
required = ['umkm_id','monthly_revenue','transaction_count','avg_historical_rating','repeat_order_rate_pct','digital_adoption_score','business_class']
required_nulls = transformed.filter(F.greatest(*[F.col(c).isNull().cast('int') for c in required]) == 1).count()
invalid_ranges = transformed.filter((F.col('monthly_revenue') <= 0) | (F.col('transaction_count') <= 0) | (~F.col('avg_historical_rating').between(1,5)) | (~F.col('repeat_order_rate_pct').between(0,100)) | (~F.col('digital_adoption_score').between(0,10))).count()
print({'rows': row_count, 'duplicates': duplicate_ids, 'required_nulls': required_nulls, 'invalid_ranges': invalid_ranges})
assert row_count >= 100000 and duplicate_ids == 0 and required_nulls == 0 and invalid_ranges == 0

In [ ]:
# Minimal tiga agregasi
by_class = transformed.groupBy('business_class').agg(F.count('*').alias('umkm_count'), F.round(F.avg('monthly_revenue'),2).alias('avg_revenue'), F.round(F.avg('net_profit_margin_pct'),2).alias('avg_margin')).orderBy(F.desc('avg_revenue'))
by_tenure = transformed.groupBy('tenure_band').agg(F.count('*').alias('umkm_count'), F.round(F.avg('repeat_order_rate_pct'),2).alias('avg_repeat_order'))
by_adoption = transformed.groupBy('digital_adoption_band').agg(F.count('*').alias('umkm_count'), F.round(F.avg('repeat_order_rate_pct'),2).alias('avg_repeat_order'), F.round(F.avg('sentiment_score'),3).alias('avg_sentiment'))
by_class.show(); by_tenure.show(); by_adoption.show()
transformed.explain('formatted')

`tenure_band` adalah cohort umur usaha, bukan urutan waktu kalender.